# 第5节：FFmpeg CLI 与媒体排查基础

本 Notebook 包含四个实验，帮助你掌握 FFmpeg 工具家族的核心用法。

**实验内容：**
1. ffprobe 深度解析
2. 查看像素格式
3. ffmpeg 抽帧与抽音频
4. 关键帧与 seek 行为分析

## 环境准备

确保已安装 ffmpeg、ffprobe、ffplay。

In [ ]:
import subprocess
import json
import os
import time

# 检查 ffmpeg、ffprobe 是否可用
def check_command(cmd):
    try:
        result = subprocess.run([cmd, '-version'], capture_output=True, text=True, timeout=10)
        version = result.stdout.split('\n')[0]
        print(f"✓ {cmd} 已安装: {version}")
        return True
    except FileNotFoundError:
        print(f"✗ {cmd} 未安装")
        return False
    except subprocess.TimeoutExpired:
        print(f"✗ {cmd} 执行超时")
        return False

check_command('ffmpeg')
check_command('ffprobe')

## 生成测试素材

生成带明显 GOP 结构的测试视频（每秒1个关键帧）。

In [ ]:
def run_ffmpeg_cmd(cmd, description, timeout=30):
    """执行 ffmpeg 命令并检查结果"""
    print(f"  {description}...")
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"  ✗ 失败: {result.stderr[:200]}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"  ✗ 超时 ({timeout}秒)")
        return False

# 生成测试视频
print("生成测试素材中...")

cmd = [
    'ffmpeg',
    '-f', 'lavfi', '-i', 'testsrc=duration=5:size=1280x720:rate=30',
    '-f', 'lavfi', '-i', 'sine=frequency=440:duration=5',
    '-c:v', 'libx264', '-g', '30', '-keyint_min', '30',
    '-c:a', 'aac', '-shortest', '-y', 'test_keyframes.mp4'
]

if run_ffmpeg_cmd(cmd, "生成测试视频"):
    size = os.path.getsize('test_keyframes.mp4') / 1024
    print(f"\n✓ 测试视频生成成功: {size:.1f} KB")

## 实验1：ffprobe 深度解析

**目标**：掌握 ffprobe 的各种输出格式和过滤器

In [ ]:
def probe_detailed(file_path):
    """使用 ffprobe 获取详细的媒体信息"""
    cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-print_format', 'json',
        '-show_format',
        '-show_streams',
        '-select_streams', 'v:0',
        '-show_entries', 'frame=pict_type,pts_time',
        '-read_intervals', '%+5',  # 只读取前5秒
        file_path
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode != 0:
            print(f"ffprobe 失败: {result.stderr[:200]}")
            return None
        return json.loads(result.stdout)
    except subprocess.TimeoutExpired:
        print("ffprobe 超时")
        return None

# 分析测试文件
print("=" * 50)
print("ffprobe 深度解析")
print("=" * 50)

info = probe_detailed('test_keyframes.mp4')
if info:
    # 显示格式信息
    fmt = info['format']
    print(f"\n容器格式: {fmt['format_name']}")
    print(f"时长: {float(fmt['duration']):.2f} 秒")
    print(f"总码率: {int(fmt['bit_rate']) / 1000:.0f} kbps")
    
    # 显示视频流信息
    for stream in info['streams']:
        if stream['codec_type'] == 'video':
            print(f"\n视频流:")
            print(f"  编码: {stream['codec_name']}")
            print(f"  分辨率: {stream['width']}x{stream['height']}")
            print(f"  帧率: {stream['r_frame_rate']}")
            print(f"  像素格式: {stream.get('pix_fmt', 'N/A')}")
    
    # 分析帧信息
    if 'frames' in info:
        frames = info['frames']
        i_frames = [f for f in frames if f.get('pict_type') == 'I']
        print(f"\n帧统计 (前5秒):")
        print(f"  总帧数: {len(frames)}")
        print(f"  关键帧数: {len(i_frames)}")

## 实验2：查看像素格式

**目标**：了解视频的像素格式详情

In [ ]:
def get_pixel_format(file_path):
    """获取视频的像素格式信息"""
    cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-select_streams', 'v:0',
        '-show_entries', 'stream=pix_fmt,pix_fmt_tags',
        '-of', 'json',
        file_path
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        if result.returncode != 0:
            print(f"ffprobe 失败: {result.stderr[:200]}")
            return None
        return json.loads(result.stdout)
    except subprocess.TimeoutExpired:
        print("ffprobe 超时")
        return None

# 常见像素格式说明
PIXEL_FORMATS = {
    'yuv420p': 'YUV 4:2:0 平面存储，视频编码标准输入',
    'yuv422p': 'YUV 4:2:2 平面存储，专业视频',
    'yuv444p': 'YUV 4:4:4 平面存储，高质量视频',
    'rgb24': 'RGB 24位，图像处理',
    'bgr24': 'BGR 24位，OpenCV 默认',
}

print("=" * 50)
print("查看像素格式")
print("=" * 50)

info = get_pixel_format('test_keyframes.mp4')
if info and 'streams' in info:
    for stream in info['streams']:
        pix_fmt = stream.get('pix_fmt', 'N/A')
        print(f"\n像素格式: {pix_fmt}")
        if pix_fmt in PIXEL_FORMATS:
            print(f"说明: {PIXEL_FORMATS[pix_fmt]}")
        else:
            print(f"说明: {stream.get('pix_fmt_tags', 'N/A')}")

## 实验3：ffmpeg 抽帧与抽音频

**目标**：掌握从视频中提取帧和音频的方法

In [ ]:
def extract_frames(input_file, output_dir, frame_type='all', timeout=30):
    """从视频中提取帧"""
    os.makedirs(output_dir, exist_ok=True)
    
    if frame_type == 'keyframe':
        # 只提取关键帧
        cmd = [
            'ffmpeg', '-i', input_file,
            '-vf', 'select=eq(pict_type\\,I)',
            '-vsync', 'vfr',
            '-y', f'{output_dir}/frame_%03d.png'
        ]
    else:
        # 提取所有帧
        cmd = [
            'ffmpeg', '-i', input_file,
            '-y', f'{output_dir}/frame_%04d.png'
        ]
    
    start = time.time()
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        elapsed = time.time() - start
        
        if result.returncode != 0:
            print(f"提取帧失败: {result.stderr[:200]}")
            return 0, elapsed
        
        # 统计提取的帧数
        frame_count = len([f for f in os.listdir(output_dir) if f.endswith('.png')])
        return frame_count, elapsed
    except subprocess.TimeoutExpired:
        print(f"提取帧超时 ({timeout}秒)")
        return 0, timeout

def extract_audio(input_file, output_file, timeout=30):
    """从视频中提取音频"""
    cmd = [
        'ffmpeg', '-i', input_file,
        '-vn',
        '-c:a', 'pcm_s16le',
        '-y', output_file
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"提取音频失败: {result.stderr[:200]}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"提取音频超时 ({timeout}秒)")
        return False

# 实验：提取关键帧
print("=" * 50)
print("提取关键帧")
print("=" * 50)

frame_count, elapsed = extract_frames('test_keyframes.mp4', 'keyframes', 'keyframe')
print(f"\n提取关键帧: {frame_count} 帧，耗时 {elapsed:.3f} 秒")

# 实验：提取所有帧
print("\n" + "=" * 50)
print("提取所有帧")
print("=" * 50)

frame_count, elapsed = extract_frames('test_keyframes.mp4', 'all_frames', 'all')
print(f"\n提取所有帧: {frame_count} 帧，耗时 {elapsed:.3f} 秒")

# 实验：提取音频
print("\n" + "=" * 50)
print("提取音频")
print("=" * 50)

if extract_audio('test_keyframes.mp4', 'output.wav'):
    size = os.path.getsize('output.wav') / 1024
    print(f"\n提取音频: {size:.1f} KB")

## 实验4：关键帧与 seek 行为分析

**目标**：理解关键帧对 seek 的影响

In [ ]:
def test_seek(input_file, seek_time, mode='input'):
    """测试 seek 行为"""
    output_file = f'seek_{mode}_{seek_time}s.png'
    
    if mode == 'input':
        # 输入 seek（-ss 放在 -i 前）：快速但不精确
        cmd = [
            'ffmpeg',
            '-ss', str(seek_time),
            '-i', input_file,
            '-vframes', '1',
            '-y', output_file
        ]
    else:
        # 输出 seek（-ss 放在 -i 后）：慢速但精确
        cmd = [
            'ffmpeg',
            '-i', input_file,
            '-ss', str(seek_time),
            '-vframes', '1',
            '-y', output_file
        ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode != 0:
            print(f"Seek 失败: {result.stderr[:200]}")
            return None
        return output_file
    except subprocess.TimeoutExpired:
        print(f"Seek 超时")
        return None

def get_frame_pts(file_path, timeout=10):
    """获取帧的时间戳"""
    cmd = [
        'ffprobe',
        '-v', 'quiet',
        '-select_streams', 'v:0',
        '-show_entries', 'frame=pts_time,pict_type',
        '-read_intervals', '%+5',  # 只读取前5秒
        '-of', 'json',
        file_path
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            return None
        return json.loads(result.stdout)
    except subprocess.TimeoutExpired:
        print("ffprobe 超时")
        return None

# 实验：分析关键帧位置
print("=" * 50)
print("分析关键帧位置")
print("=" * 50)

frames_info = get_frame_pts('test_keyframes.mp4')
if frames_info and 'frames' in frames_info:
    frames = frames_info['frames']
    i_frames = [f for f in frames if f.get('pict_type') == 'I']
    
    print(f"\n总帧数 (前5秒): {len(frames)}")
    print(f"关键帧时间点:")
    for f in i_frames[:5]:  # 只显示前5个
        print(f"  I帧: {float(f['pts_time']):.3f} 秒")

# 实验：测试 seek 行为
print("\n" + "=" * 50)
print("测试 seek 行为")
print("=" * 50)

# 测试 seek 到 2.5 秒（非关键帧时间点）
seek_time = 2.5

# 输入 seek（快速但不精确）
output1 = test_seek('test_keyframes.mp4', seek_time, 'input')
print(f"\n输入 seek 到 {seek_time} 秒: {output1}")
print("  特点: 快速，直接跳到最近关键帧，可能不精确")

# 输出 seek（慢速但精确）
output2 = test_seek('test_keyframes.mp4', seek_time, 'output')
print(f"\n输出 seek 到 {seek_time} 秒: {output2}")
print("  特点: 精确，从头解码到目标帧，速度较慢")

# 比较两种 seek 的结果
if output1 and output2:
    size1 = os.path.getsize(output1) / 1024
    size2 = os.path.getsize(output2) / 1024
    print(f"\n文件大小对比:")
    print(f"  输入 seek: {size1:.1f} KB")
    print(f"  输出 seek: {size2:.1f} KB")

## 总结

通过本实验，你应该掌握了：

1. **FFmpeg 工具家族**
   - `ffmpeg`：音视频处理（转码、转封装、抽帧、抽音频）
   - `ffprobe`：媒体文件探测（容器信息、流信息、时间戳）
   - `ffplay`：轻量播放器（快速验证、调试滤镜）

2. **ffmpeg 常用操作**
   - `-c copy`：转封装模式，速度快
   - `-vf "select=eq(pict_type\,I)"`：提取关键帧
   - `-vn`：不包含视频流

3. **关键帧与 seek**
   - 关键帧是完整的图像，可以独立解码
   - seek 只能精确到关键帧
   - `-ss` 放在 `-i` 前后有不同行为：
     - 输入 seek：快速但不精确
     - 输出 seek：慢速但精确